In [ ]:
## I. Ingestion and Transformation

In [ ]:
test

Historical data into panda dataframes - extract all data into datframes

In [ ]:
import pandas as pd
import os

os.chdir('/Users/noahg/Library/CloudStorage/OneDrive-Persönlich/Dokumente/Uni/Master_MMT_TUM/Semester_2/Python_for_Life_Science/Python_For_Lifescience_Project/PythonProject/src/historicals')

# Control of Corruption Index
ControlOfCorruption_df = pd.read_csv("ControlOfCorruption_Historical_WorldBank_1996-2023.csv",sep=",",skiprows=3)

# Employment in Agriculture 
EmploymentInAgriculture_df = pd.read_csv("/Users/noahg/Library/CloudStorage/OneDrive-Persönlich/Dokumente/Uni/Master_MMT_TUM/Semester_2/Python_for_Life_Science/Python_For_Lifescience_Project/PythonProject/src/historicals/EmploymentInAgriculture_Historical_WorldBank_1991-2023.csv",sep=",", skiprows=3)


# GDP 
GDP_df = pd.read_csv("/Users/noahg/Library/CloudStorage/OneDrive-Persönlich/Dokumente/Uni/Master_MMT_TUM/Semester_2/Python_for_Life_Science/Python_For_Lifescience_Project/PythonProject/src/historicals/GDP_HistoricalData_1960_2024.csv",sep=",", skiprows=3)

# HDI
HDI_df = pd.read_csv("/Users/noahg/Library/CloudStorage/OneDrive-Persönlich/Dokumente/Uni/Master_MMT_TUM/Semester_2/Python_for_Life_Science/Python_For_Lifescience_Project/PythonProject/src/historicals/HDI_1990_2019.csv",sep=",")

Clean data according to chosen temporal range (1995-2024) 

In [157]:
# Control of Corruption, Employment in Agriculture, GDP, Gini Coefficient data all come from World Bank and the CSVs are structured similiariy. 
# we therefore can perform the data cleaning in a loop for these 


dfs = [ControlOfCorruption_df, EmploymentInAgriculture_df, GDP_df]

for index, df in enumerate(dfs): 
    cols_to_drop = df.loc[:, 'Country Code' : '1992'].columns
    df_current = df.drop(columns=cols_to_drop)

    na_per_row = df_current.isna().sum(axis=1)

    df_current.insert(1,'na_count',na_per_row)

    df_current = df_current[df_current['na_count']<=10]
    df_current = df_current.reset_index()
    df_current = df_current.drop('index', axis=1)
    df_current = df_current.drop('2024', axis=1)
    dfs[index] = df_current


In [ ]:
import numpy as np
from scipy.stats import gmean

# Now we apply similiar cleanup-logic to the HumanDevelopmentIndex dataframe

HDI_df = HDI_df.drop('HDI Rank', axis=1)
cols_to_drop = df.loc[:, '1990':'1992']
HDI_df = HDI_df.drop(columns=cols_to_drop)
HDI_df.rename(columns={'Country':'Country Name'}, inplace=True)


# HDI_df = HDI_df.replace('..',np.nan)
cols_to_fix = HDI_df.columns[3:]

HDI_df[cols_to_fix] = HDI_df[cols_to_fix].apply(pd.to_numeric,errors = 'coerce') 


na_per_row = HDI_df.isna().sum(axis=1)
HDI_df.insert(1,'na_count',na_per_row)
HDI_df = HDI_df[HDI_df['na_count']<=10]
HDI_df = HDI_df.reset_index()

# extrapolate HDI to match the temporal range of the other dataframes. 
# was 1993 - 2019 and the other dataframes 1993 - 2023
# using geometrical mean over the last 5 years (2015 - 2019) to account for compunding effects


HDI_df['Growth Factor'] = (HDI_df.iloc[:,-1] / HDI_df.iloc[:,-5])**(1/5)

growth_col = HDI_df.pop('Growth Factor')

HDI_df.insert(1,'Growth Factor',growth_col)



for i in ('2020', '2021', '2022', '2023'):
    prev_year = str(int(i)-1)
    HDI_df[i] = HDI_df[prev_year] *  HDI_df['Growth Factor']
    


print(HDI_df.iloc[:,-5:])




      2019      2020      2021      2022      2023
0    0.511  0.513229  0.515467  0.517716  0.519974
1    0.795  0.796407  0.797817  0.799230  0.800645
2    0.748  0.749610  0.751224  0.752841  0.754462
3    0.868  0.869205  0.870412  0.871620  0.872830
4    0.581  0.582817  0.584640  0.586468  0.588302
..     ...       ...       ...       ...       ...
180  0.711  0.699936  0.689044  0.678321  0.667766
181  0.704  0.707244  0.710504  0.713778  0.717067
182  0.470  0.467442  0.464899  0.462369  0.459852
183  0.584  0.587047  0.590110  0.593189  0.596284
184  0.571  0.574670  0.578363  0.582080  0.585821

[185 rows x 5 columns]
